# Category data

In [1]:
import numpy as np
import pandas as pd
import sys
print(sys.platform, sys.version, f'numpy version {np.__version__}',  f'pandas version {pd.__version__}',sep='\n')

win32
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
numpy version 2.4.1
pandas version 3.0.0


In [2]:
from pathlib import Path
d = pd.read_csv(Path('prepare', 'data4.csv'), encoding='utf-8')
d.head(3)

,group,name,part1,part2,total,sum
0,G1,Студент № 1,7,16,23,23
1,G1,Студент № 2,10,20,30,30
2,G1,Студент № 3,10,12,22,22


In [3]:
d.group.unique()

<StringArray>
['G1', 'G2']
Length: 2, dtype: str

Документація на метод <code>nlargest</code>  <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html#pandas.DataFrame.nlargest" target=_blank rel="noopener noreferrer">тут</a>.
Зверніть увагу на параметр <code>keep</code>.

In [4]:
d.loc[d.total.nlargest(2, keep='all').index, :]

,group,name,part1,part2,total,sum
1,G1,Студент № 2,10,20,30,30
3,G1,Студент № 4,10,28,30,38
6,G1,Студент № 7,10,22,30,32
9,G1,Студент № 10,10,24,30,34
13,G1,Студент № 14,10,24,30,34
20,G2,Студент № 21,10,26,30,36
22,G2,Студент № 23,10,20,30,30
25,G2,Студент № 26,10,24,30,34
29,G2,Студент № 30,10,20,30,30


In [5]:
d.describe(include='all')

,group,name,part1,part2,total,sum
count,31,31,31.000000,31.000000,31.000000,31.000000
unique,2,31,NaN,NaN,NaN,NaN
top,G1,Студент № 1,NaN,NaN,NaN,NaN
freq,16,1,NaN,NaN,NaN,NaN
mean,NaN,NaN,9.225806,15.032258,23.354839,24.258065
std,NaN,NaN,1.745655,7.739440,7.709511,8.721497
min,NaN,NaN,3.000000,0.000000,7.000000,7.000000
25%,NaN,NaN,10.000000,12.000000,20.000000,20.000000
50%,NaN,NaN,10.000000,16.000000,26.000000,26.000000
75%,NaN,NaN,10.000000,20.000000,30.000000,30.000000


Для нечислових даних статистична інформація відображає кількість унікальних значень.

In [6]:
d['group'].unique()

<StringArray>
['G1', 'G2']
Length: 2, dtype: str

### Way 1: obvious
Map to codes, add new column with codes, drop old column with objects

In [7]:
d1 = d.copy()
groups = {'G1':1, 'G2':2}
d1['group_code'] = d1['group'].map(groups)
d1[::8]

,group,name,part1,part2,total,sum,group_code
0,G1,Студент № 1,7,16,23,23,1
8,G1,Студент № 9,5,4,9,9,1
16,G2,Студент № 17,7,0,7,7,2
24,G2,Студент № 25,10,8,18,18,2


In [8]:
d1.drop(columns=['group'], inplace=True)
d1.head(3)

,name,part1,part2,total,sum,group_code
0,Студент № 1,7,16,23,23,1
1,Студент № 2,10,20,30,30,1
2,Студент № 3,10,12,22,22,1


In [9]:
d1.rename(columns={'group_code':'group'}, inplace=True)
d1.head(3)

,name,part1,part2,total,sum,group
0,Студент № 1,7,16,23,23,1
1,Студент № 2,10,20,30,30,1
2,Студент № 3,10,12,22,22,1


### Way 2: also obvious
Map to codes, replace column with objects by column with codes

In [10]:
d2 = d.copy()
groups = {'G1':1, 'G2':2}
d2['group'] = d2['group'].map(groups)
d2[::8]

,group,name,part1,part2,total,sum
0,1,Студент № 1,7,16,23,23
8,1,Студент № 9,5,4,9,9
16,2,Студент № 17,7,0,7,7
24,2,Студент № 25,10,8,18,18


### Way 3: auto convert to the categorical type

In [11]:
d3 = d.copy()
d3['group_ct'] = d3.group.astype('category') # here was added new column, but this is not nessesary
d3[::8]

,group,name,part1,part2,total,sum,group_ct
0,G1,Студент № 1,7,16,23,23,G1
8,G1,Студент № 9,5,4,9,9,G1
16,G2,Студент № 17,7,0,7,7,G2
24,G2,Студент № 25,10,8,18,18,G2


In [12]:
d3.group_ct.dtype

CategoricalDtype(categories=['G1', 'G2'], ordered=False, categories_dtype=str)

In [13]:
d3.group_ct.head(3)

0    G1
1    G1
2    G1
Name: group_ct, dtype: category
Categories (2, str): ['G1', 'G2']

In [14]:
d3.loc[0, 'group_ct']

'G1'

### Way 4: convert to the user defined categorical type
convert type at loading is also possible

In [15]:
t = pd.CategoricalDtype(categories=['G2', 'G1'], ordered=True)
t

CategoricalDtype(categories=['G2', 'G1'], ordered=True, categories_dtype=str)

In [16]:
d4 = d.copy()
d4['group_ct'] = d4.group.astype(t)
d4.head(3)

,group,name,part1,part2,total,sum,group_ct
0,G1,Студент № 1,7,16,23,23,G1
1,G1,Студент № 2,10,20,30,30,G1
2,G1,Студент № 3,10,12,22,22,G1


In [17]:
d4.sort_values(by='group_ct').head(3)

,group,name,part1,part2,total,sum,group_ct
30,G2,Студент № 31,10,18,28,28,G2
28,G2,Студент № 29,7,4,11,11,G2
27,G2,Студент № 28,10,0,10,10,G2


In [18]:
d4.group_ct.head(3)

0    G1
1    G1
2    G1
Name: group_ct, dtype: category
Categories (2, str): ['G2' < 'G1']

In [19]:
d4.describe(include='category')

,group_ct
count,31
unique,2
top,G1
freq,16


In [20]:
d4.to_csv(Path('prepare', 'data5.csv'), index=False)

In [21]:
d5 = pd.read_csv(Path('prepare', 'data5.csv'), 
                 dtype={'group':pd.CategoricalDtype(categories=['G1', 'G2'], ordered=True)})
d5.head(3)

,group,name,part1,part2,total,sum,group_ct
0,G1,Студент № 1,7,16,23,23,G1
1,G1,Студент № 2,10,20,30,30,G1
2,G1,Студент № 3,10,12,22,22,G1


In [22]:
d5.dtypes

group       category
name             str
part1          int64
part2          int64
total          int64
sum            int64
group_ct         str
dtype: object